# v3.0 坏数据取证 (C13 - C16)

接 `01_quality_audit.ipynb`。品种筛查已定（57 保留 / 22 剔除），这个 notebook 回答
**保留下来的品种里还剩什么坏数据、以及能不能填补**。

| 编号 | 检查项 | 回答什么问题 |
|---|---|---|
| C13 | 0 价 bar 逐行取证 | OHLC 里有 1 个 / 多个 / 全部为 0？能不能填？ |
| C14 | 分钟级可交易性 | 去掉假 session 之后，真正有成交的分钟占比、最长停滞段 |
| C15 | ATR 中毒 | ATR(14) 恰好为 0 的 bar 数：未清洗 vs 清洗后 |
| C16 | 日内活跃度双峰检测 | **替代「日成交量中位数」的筛选指标** |

实测结论速览：
- **0 价：没有「只有 1 个为 0」的情况。** 5 根是 `open+low` 同时为 0（可填，全在 21:01），
  91 根是 4 个全为 0（不可填）。在 57 个保留品种里只剩 **B 一个品种、共 91 根**。
- **ATR(14)==0 在 1m 上有 1,113,051 根，清洗只降到 691,703（-38%）**，TS 完全没降 ——
  **数据清洗解决不了这个问题，必须在风险层夹逼分母。**
- **B 是完美双峰**：中位数日 99.2% 的分钟有成交，但 43.4% 的交易日基本不交易。
  「日成交量中位数」这个指标恰好被双峰骗过去了。

In [ ]:
# ============================ §1 配置 ============================
import time

import numpy as np
import pandas as pd

PKL = r"D:\2026_Summer\TradingApp\data\v3.0\all_symbol_min_full_main_close_k_1.pkl"
OUT = r"D:\2026_Summer\TradingApp\data\v3.0"

# 01_quality_audit.ipynb 的输出
KEEP = ['A', 'AG', 'AL', 'AP', 'AU', 'B', 'BU', 'C', 'CF', 'CJ', 'CS', 'CU', 'EB', 'EG', 'FG',
        'FU', 'HC', 'I', 'IC', 'IF', 'IH', 'J', 'JD', 'JM', 'L', 'LH', 'LU', 'M', 'MA', 'NI',
        'NR', 'OI', 'P', 'PB', 'PF', 'PG', 'PK', 'PP', 'RB', 'RM', 'RU', 'SA', 'SC', 'SF',
        'SM', 'SN', 'SP', 'SR', 'SS', 'T', 'TA', 'TF', 'TS', 'UR', 'V', 'Y', 'ZN']

# C16 的候选筛选阈值（替代「日成交量中位数」）
TH_DAY_TRADED_FRAC = 0.50   # 一个交易日里「有成交分钟占比」低于这个 = 死日
TH_MAX_DEAD_DAY_PCT = 0.05  # 死日占比超过这个 = 不适合做日内策略

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 400)
print("配置就绪")

In [ ]:
# ============================ §2 加载 + 基础量 ============================
t0 = time.time()
df = pd.read_pickle(PKL)
print(f"[{time.time() - t0:5.1f}s] loaded {df.shape}")

sym_codes, SYMS = pd.factorize(df["underlying_symbol"])
sym_codes = sym_codes.astype(np.int32)
NS = len(SYMS)

o = df["open"].to_numpy()
h = df["high"].to_numpy()
lo = df["low"].to_numpy()
c = df["close"].to_numpy()
v = df["volume"].to_numpy()
oi = df["open_interest"].to_numpy()

IDX_NS = df.index.values.astype("int64")
CAL_D = (IDX_NS // 86_400_000_000_000).astype(np.int32)
CD0, NCD = int(CAL_D.min()), int(CAL_D.max()) - int(CAL_D.min()) + 1

SAME_PREV = np.empty(len(df), bool)
SAME_PREV[0] = False
SAME_PREV[1:] = sym_codes[1:] == sym_codes[:-1]
DT_NS = np.empty(len(df), "int64")
DT_NS[0] = 0
DT_NS[1:] = np.diff(IDX_NS)

# 假日历日（整个日历日成交量合计为 0）—— C6 的定义，这里复用
KEY_CD = sym_codes.astype(np.int64) * NCD + (CAL_D - CD0)
fake_cd = ((np.bincount(KEY_CD, minlength=NS * NCD) > 0)
           & (np.bincount(KEY_CD, weights=v, minlength=NS * NCD) == 0))
ROW_FAKE = fake_cd[KEY_CD]
print(f"[{time.time() - t0:5.1f}s] 假日历日行数 = {int(ROW_FAKE.sum()):,} ({ROW_FAKE.mean():.2%})")

## C13  0 价 bar 逐行取证

**可填补性完全取决于「OHLC 里有几个是 0」**：

- **1 个为 0** → 其余 3 个仍在，且 OHLC 之间有约束（`low <= min(o,c) <= max(o,c) <= high`），
  有填补空间
- **2~3 个为 0** → 难填，要看是哪几个
- **4 个全为 0** → 这根 bar 没有任何价格信息，**无法填补**，只能丢弃（或用
  prev_close/next_open 夹逼，但那是编造）

另外两个正交维度：
- `volume > 0` → **真的有成交但价格丢了**（真实数据损失）；`volume == 0` → 只是个空 bar
- 是否落在假日历日里 → 如果是，走 C6 的整段丢弃逻辑就顺手解决了

In [ ]:
zo, zh, zl, zc = (o == 0), (h == 0), (lo == 0), (c == 0)
nzero = zo.astype(np.int8) + zh + zl + zc
anyzero = nzero > 0
idx = np.flatnonzero(anyzero)

print("=" * 100, f"\nC13  0 价 bar 取证 —— 共 {len(idx)} 根\n", "=" * 100)
print("\n--- 按「OHLC 里有几个 0」分类 ---")
LABEL = {1: "其余 3 个仍在 -> **有填补空间**",
         2: "-> 难填，看是哪两个",
         3: "-> 难填",
         4: "无任何价格信息 -> **无法填补**"}
for n in range(1, 5):
    m = nzero == n
    print(f"\n  {n} 个为 0: {int(m.sum()):>4} 根   {LABEL[n]}")
    if m.any():
        print(f"       品种分布 : {df.loc[m, 'underlying_symbol'].value_counts().to_dict()}")
        print(f"       volume>0 : {int((m & (v > 0)).sum())} 根（真有成交，价格丢了）"
              f"    volume==0: {int((m & (v == 0)).sum())} 根")
        print(f"       落在假日历日里: {int((m & ROW_FAKE).sum())} 根"
              f"    在 KEEP 名单里的: {int((m & df['underlying_symbol'].isin(KEEP).to_numpy()).sum())} 根")

print("\n--- 具体是哪几个字段为 0（combo）---")
combo = (np.where(zo, "O", "") + np.where(zh, "H", "")
         + np.where(zl, "L", "") + np.where(zc, "C", ""))[anyzero]
cb = pd.DataFrame({"combo": combo,
                   "sym": df.loc[anyzero, "underlying_symbol"].to_numpy(),
                   "vol_gt0": v[idx] > 0,
                   "tod": pd.to_datetime(df.index[idx]).strftime("%H:%M")})
print(cb.groupby(["combo", "sym"]).agg(
    n=("vol_gt0", "size"), n_vol_gt0=("vol_gt0", "sum"),
    时刻=("tod", lambda s: " ".join(sorted(set(s))[:6]))).to_string())

In [ ]:
# 逐行明细 + 前后邻居（判断能不能夹逼填补）
nxt = np.minimum(idx + 1, len(df) - 1)
nxt_ok = (idx + 1 < len(df)) & SAME_PREV[nxt]
detail = pd.DataFrame({
    "datetime": df.index[idx],
    "sym": df["underlying_symbol"].to_numpy()[idx],
    "contract": df["contract"].to_numpy()[idx],
    "n_zero": nzero[idx], "which": combo,
    "open": o[idx], "high": h[idx], "low": lo[idx], "close": c[idx],
    "volume": v[idx], "open_interest": oi[idx],
    "in_fake_day": ROW_FAKE[idx],
    "in_KEEP": df["underlying_symbol"].isin(KEEP).to_numpy()[idx],
    "prev_close": np.where(SAME_PREV[idx], c[idx - 1], np.nan),
    "next_open": np.where(nxt_ok, o[nxt], np.nan),
    "next_close": np.where(nxt_ok, c[nxt], np.nan),
})
detail.to_csv(rf"{OUT}\audit_C13_zero_price_rows.csv", index=False)

print("--- 0 价 bar 是否连成整段（gap>1min 断开）---")
for s, grp in detail.groupby("sym"):
    t = pd.Series(grp["datetime"].to_numpy())
    runs = (t.diff() > pd.Timedelta("1min")).cumsum()
    keep_tag = "【KEEP】" if s in KEEP else "【已剔除】"
    print(f"\n  {s} {keep_tag}: {len(grp)} 根 -> {runs.nunique()} 段")
    for _, gg in t.groupby(runs):
        sub = grp.iloc[gg.index]
        print(f"      {gg.iloc[0]} -> {gg.iloc[-1]}  ({len(gg)} 根, "
              f"vol合计={sub['volume'].sum():.0f}, which={sub['which'].iloc[0]}, "
              f"prev_close={sub['prev_close'].iloc[0]}, next_close={sub['next_close'].iloc[-1]})")

print("\n--- 只看 KEEP 名单里的 0 价 bar 明细 ---")
print(detail[detail["in_KEEP"]].to_string(index=False))

### C13 处置建议

| combo | 根数 | 可填补性 | 建议 |
|---|---|---|---|
| `OL`（open+low 为 0，high/close 有效） | 5 | **可填** | `open := 前一根 close`；`low := min(high, close, 前一根close)`。全部位于 **21:01**（夜盘竞价 bar 之后第一个真实分钟）—— 系统性供应商 bug，不是随机损坏，所以规则化填补是安全的 |
| `OHLC`（4 个全 0） | 91 | **不可填** | 整根丢弃。B 的 90 根是**一整个下午盘连续缺失**（2017-05-12 13:31→15:00），必须**整段 session 丢弃**，不能只丢单根 |

注意：没有「只有 1 个为 0」的情况，所以「利用 OHLC 内部约束反推」这条路用不上。

## C14  分钟级可交易性

`traded_pct` = 真实 session（已排除假日历日）里 `volume > 0` 的 bar 占比。
这个指标决定**一个品种能不能跑分钟级策略** —— 一根没有成交的 bar 上的任何成交假设都是虚构的。

`max_stale_run` = 一个 session 内最长的连续零成交分钟数。它直接告诉你指标最坏会吃进
多长的停滞价格。

In [ ]:
zv = (v == 0) & ~ROW_FAKE
brk = np.empty(len(df), bool)
brk[0] = True
brk[1:] = (~SAME_PREV[1:]) | (DT_NS[1:] != 60_000_000_000) | ROW_FAKE[1:] | ROW_FAKE[:-1]

x = zv.astype(np.int8)
newrun = np.empty(len(df), bool)
newrun[0] = True
newrun[1:] = (x[1:] != x[:-1]) | brk[1:]
run_len = np.bincount(np.cumsum(newrun) - 1)
first_i = np.flatnonzero(newrun)
is_zero_run = x[first_i] == 1
run_sym = sym_codes[first_i]

max_stale = np.zeros(NS, np.int64)
np.maximum.at(max_stale, run_sym[is_zero_run], run_len[is_zero_run])
p99_stale = (pd.Series(run_len[is_zero_run]).groupby(run_sym[is_zero_run])
             .quantile(0.99).reindex(range(NS)).to_numpy())

# session 最长连续长度（不看成交）—— 用来验证 max_stale 是否自洽
sbrk = np.empty(len(df), bool)
sbrk[0] = True
sbrk[1:] = (~SAME_PREV[1:]) | (DT_NS[1:] != 60_000_000_000)
slen = np.bincount(np.cumsum(sbrk) - 1)
sfirst = np.flatnonzero(sbrk)
max_session = np.zeros(NS, np.int64)
np.maximum.at(max_session, sym_codes[sfirst], slen)

real = ~ROW_FAKE
n_real = np.bincount(sym_codes[real], minlength=NS)
n_traded = np.bincount(sym_codes[real & (v > 0)], minlength=NS)
n_all = np.bincount(sym_codes, minlength=NS)

C14 = pd.DataFrame({
    "n_bars": n_all, "n_real": n_real,
    "fake_pct": 1 - n_real / np.maximum(n_all, 1),
    "traded_pct": n_traded / np.maximum(n_real, 1),
    "max_stale_run": max_stale, "p99_stale_run": p99_stale,
    "max_session_len": max_session,
}, index=pd.Index(SYMS, name="sym")).loc[lambda d: d.index.isin(KEEP)]
C14["stale_is_full_session"] = C14["max_stale_run"] == C14["max_session_len"]

print("=" * 100, "\nC14  分钟级可交易性（按 traded_pct 升序）\n", "=" * 100)
print(C14.sort_values("traded_pct").to_string())
print("\nmax_stale_run == max_session_len 的品种（= 存在「一整个 session 零成交」）:")
print("  " + " ".join(sorted(C14.index[C14["stale_is_full_session"]])))
C14.to_csv(rf"{OUT}\audit_C14_tradability.csv")

## C15  ATR 中毒实测

**`ATR == 0` 是最危险的后果**：止损距离 = `sl_atr_mult * ATR` → 0，
仓位 = `risk / 止损距离` → **无穷大**。

这里对每个 KEEP 品种，在 1m 序列上算 ATR(14)，分别统计**未清洗**和**删掉假日历日之后**
恰好等于 0 的 bar 数。

⚠️ 现有策略的风险 ATR 读的是 **30m**（`config.risk_atr_on_30m=True`），不是 1m。
所以下面同时给 1m / 5m / 30m 三个周期，看严重度随周期怎么衰减。

In [ ]:
def atr_zero_stats(H, L, C, period=14):
    """返回 (ATR==0 的 bar 数, ATR 中位数)。TR 用标准定义，首根退化为 H-L。"""
    if len(C) < period + 6:
        return 0, np.nan
    pc = np.concatenate(([np.nan], C[:-1]))
    tr = np.maximum.reduce([H - L, np.abs(H - pc), np.abs(L - pc)])
    tr[0] = H[0] - L[0]
    a = pd.Series(tr).rolling(period).mean().to_numpy()
    fin = np.isfinite(a)
    return int((a[fin] == 0).sum()), float(np.nanmedian(a))


def agg_n(H, L, C, n):
    """把 1m 序列按每 n 根一桶聚合（只用于估计高周期的 ATR==0 频率，
    不做时段内对齐 —— 这里只要量级，不要精确 bar）。"""
    m = (len(C) // n) * n
    Hn = H[:m].reshape(-1, n).max(1)
    Ln = L[:m].reshape(-1, n).min(1)
    Cn = C[:m].reshape(-1, n)[:, -1]
    return Hn, Ln, Cn


rows = []
for i, s in enumerate(SYMS):
    if s not in KEEP:
        continue
    a = int(np.argmax(sym_codes == i))
    b = len(sym_codes) - int(np.argmax((sym_codes == i)[::-1]))
    sl = slice(a, b)
    rf = ROW_FAKE[sl]
    H, L, C = h[sl], lo[sl], c[sl]
    r = {"sym": s}
    for tf, n in (("1m", 1), ("5m", 5), ("30m", 30)):
        zr, mr = atr_zero_stats(*agg_n(H, L, C, n)) if n > 1 else atr_zero_stats(H, L, C)
        zc_, mc = (atr_zero_stats(*agg_n(H[~rf], L[~rf], C[~rf], n)) if n > 1
                   else atr_zero_stats(H[~rf], L[~rf], C[~rf]))
        r[f"atr0_{tf}_RAW"] = zr
        r[f"atr0_{tf}_CLEAN"] = zc_
        r[f"atr0_{tf}_CLEAN_pct"] = zc_ / max(len(C) // n, 1)
        if tf == "1m":
            r["atr_med_RAW"], r["atr_med_CLEAN"] = mr, mc
    rows.append(r)
    print(f"..{s}", end="", flush=True)

C15 = pd.DataFrame(rows).set_index("sym")
C15["atr_med_bias"] = C15["atr_med_RAW"] / C15["atr_med_CLEAN"] - 1

print("\n\n" + "=" * 100)
print("C15  ATR(14)==0 的 bar 数")
print("=" * 100)
for tf in ("1m", "5m", "30m"):
    print(f"  {tf:>4}:  未清洗 {int(C15[f'atr0_{tf}_RAW'].sum()):>10,}  ->  "
          f"清洗后 {int(C15[f'atr0_{tf}_CLEAN'].sum()):>10,}   "
          f"(降幅 {1 - C15[f'atr0_{tf}_CLEAN'].sum() / max(C15[f'atr0_{tf}_RAW'].sum(), 1):.0%})")
print("\n--- 清洗后 30m 上 ATR==0 占比最高的 15 个品种（这才是策略真正用的周期）---")
print(C15.sort_values("atr0_30m_CLEAN_pct", ascending=False)
      [["atr0_1m_CLEAN", "atr0_1m_CLEAN_pct", "atr0_5m_CLEAN", "atr0_30m_CLEAN",
        "atr0_30m_CLEAN_pct", "atr_med_bias"]].head(15).to_string())
print("\n--- ATR 中位数被 padding 拉偏的品种（其余均为 0.0）---")
print(C15[C15["atr_med_bias"].abs() > 1e-6][["atr_med_RAW", "atr_med_CLEAN", "atr_med_bias"]]
      .sort_values("atr_med_bias").to_string())
C15.to_csv(rf"{OUT}\audit_C15_atr_zero.csv")

## C16  日内活跃度双峰检测 —— 替代「日成交量中位数」的筛选指标

**为什么「日成交量中位数」是错的指标**：它是一个中位数，对**双峰分布完全失效**。
B（豆二）的中位数日有 99.2% 的分钟成交（看起来完美），但 43.4% 的交易日基本不交易。
中位数只看到了好的那个峰。

**正确的指标看尾部**：逐交易日算「有成交分钟占比」，然后看
- `dead_day_pct` = 该占比 < 50% 的交易日比例
- `q05` / `q25` = 该占比的 5% / 25% 分位数

一个能跑日内策略的品种，应该 `q05` 就已经很高了 —— 也就是**连最差的日子都在交易**。

In [ ]:
td_all = df["trading_date"].to_numpy()
work = pd.DataFrame({
    "sym_code": sym_codes[real],
    "td": td_all[real],
    "traded": (v[real] > 0).astype(np.int8),
})
per_day = work.groupby(["sym_code", "td"])["traded"].mean()

rows = []
for i, s in enumerate(SYMS):
    if s not in KEEP:
        continue
    d = per_day.loc[i]
    rows.append({
        "sym": s, "n_days": len(d),
        "q01": d.quantile(0.01), "q05": d.quantile(0.05), "q10": d.quantile(0.10),
        "q25": d.quantile(0.25), "median": d.median(),
        "dead_day_pct": float((d < TH_DAY_TRADED_FRAC).mean()),
    })
C16 = pd.DataFrame(rows).set_index("sym")
C16["intraday_ok"] = C16["dead_day_pct"] <= TH_MAX_DEAD_DAY_PCT

print("=" * 100, "\nC16  日内活跃度（按 dead_day_pct 降序 = 最差的在最上面）\n", "=" * 100)
print(C16.sort_values("dead_day_pct", ascending=False).to_string())
C16.to_csv(rf"{OUT}\audit_C16_intraday_activity.csv")

print(f"\n--- 用新指标筛选（dead_day_pct <= {TH_MAX_DEAD_DAY_PCT:.0%}）---")
bad16 = C16.index[~C16["intraday_ok"]].tolist()
print(f"不适合日内策略: {len(bad16)} 个 -> {' '.join(sorted(bad16))}")
print(f"适合日内策略  : {int(C16['intraday_ok'].sum())} 个")
print(f"\nINTRADAY_SYMBOLS = {sorted(C16.index[C16['intraday_ok']].tolist())}")
print(f"DAILY_ONLY_SYMBOLS = {sorted(bad16)}")

print("\n--- 对照：新旧指标的分歧 ---")
cmp = C16[["median", "q05", "dead_day_pct", "intraday_ok"]].join(C14[["traded_pct", "p99_stale_run"]])
print(cmp.sort_values("q05").head(15).to_string())
print("\n注意 median 这一列：几乎所有品种都接近 1.0 —— 这就是「中位数指标」看不出问题的原因。")

print(f"\n[{time.time() - t0:5.1f}s] DONE")